# Tool 7bis — Automatic epoch rejection — Curry twin (channel-first, then epoch)

*Curry twin: functionally identical to the EDF tool 7bis. 7bis reads only tool-6 `*_all-epo.fif` + `*_epoch_channel_rejection.tsv`, which the Curry tool 6 writes in the same MNE/TSV format, so no code differs. Generated by `tools_curry/_make_tool7bis_curry.py` — do not hand-edit; edit the EDF notebook and re-run the generator.*

Automatically cleans the epochs that **6_preprocessing** flagged, using a **channel-first then epoch**
rule (à la PREP / FASTER): a globally-bad channel is dropped *before* the epoch vote, so it no longer
condemns every epoch it appears in.

**Per participant:** ① a **channel is dropped** when flagged in more than *Channel reject %* of its
in-scope epochs → ② an **epoch is rejected** when flagged in more than *Epoch reject %* of the remaining
**good** channels → ③ the clean `*_clean-epo.fif` (selected stages, kept epochs, bad channels dropped) is
written to `clean_epo_auto/` **beside the raw-epochs folder**, and the reports (decision TSV, HTML, global
summary) to `reports_rejection_auto/` **beside the reports folder** — following the toolkit's
`derivatives/` (data) vs `reports_*` (reports) split.

**Inputs (read-only):** two folders are selected **explicitly** — the **raw-epochs folder** (holding the
tool-6 `{file_id}_all-epo.fif`, e.g. `derivatives/raw_epo`) and the **reports folder** (holding
`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preprocessing`). Picking them explicitly (rather than
scanning one root recursively) keeps several tool-6 runs apart when their outputs have been renamed/versioned
(`raw_epo_v1`, `reports_preprocessing_v2`, …). Tool-6 outputs are never modified. No channel interpolation —
bad channels are dropped (interpolation is a later step).

**Workflow:** ① Select the raw-epochs + reports folders and **Scan** → ② set thresholds + stages of
interest → ③ Run.

In [ ]:
import os, sys, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from ipyfilechooser import FileChooser
import mne
mne.set_log_level('ERROR')

# Reuse the shared tool-7 library (find_participants, save_report_html, custom-stage styling). Search
# several locations so it is found whether Voila is launched from the repo root (lib in tools/) or from
# tools_curry/ (the Curry twin lives there; the shared lib stays in ../tools).
_here = os.getcwd()
_lib_candidates = [_here, os.path.join(_here, 'tools'),
                   os.path.join(os.path.dirname(_here), 'tools'), os.path.join(_here, '..', 'tools')]
for _cand in _lib_candidates:
    if os.path.isfile(os.path.join(_cand, 'qc_rejected_epochs_lib.py')):
        _cand = os.path.abspath(_cand)
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break
import qc_rejected_epochs_lib as L

AASM_STAGES = ['W', 'N1', 'N2', 'N3', 'R']

# ---- Editable defaults (channel-first, then epoch) ----
DEFAULT_CHANNEL_REJECT_PCT = 20.0   # drop a channel flagged in > this % of its in-scope epochs
DEFAULT_EPOCH_REJECT_PCT   = 20.0   # drop an epoch flagged in > this % of the remaining good channels

S = {}   # shared state across sections


def read_event_counts(path):
    """Return {canonical_type: n_events} from a tool-6 {file_id}_event_counts.tsv (raw per-type event
    count for the whole file), or {} if the sidecar is absent/unreadable. Feeds the per-event
    participant-exclusion threshold."""
    try:
        d = pd.read_csv(path, sep='\t')
        return {str(r['event_type']): int(r['n_events']) for _, r in d.iterrows()}
    except Exception:
        return {}


def build_pair_matrix(tsv_path, methods=None, event_master=True, event_types=None, event_flags_path=None):
    """Recompose a boolean (epoch x channel) rejection matrix from a tool-6
    {file_id}_epoch_channel_rejection.tsv, keeping ONLY the selected flagging methods (their per-method
    flag_<method> columns OR'd together), plus a per-epoch stage Series and a small meta dict.

    methods          : methods to include (subset of L.METHOD_ORDER); None = every method present.
    event_master     : include the 'event' method at all (only when 'event' is also in `methods`).
    event_types      : canonical event types allowed to reject (sub-selection); needs event_flags_path.
    event_flags_path : the tool-6 {file_id}_event_epoch_flags.tsv (one evt_<type> column per epoch).
                       Absent -> fall back to the TSV's OR'd flag_event (all event types, no sub-select).

    With methods=None + event_master=True the union of every flag_<method> equals the stored reject_any,
    so a default (all-methods) run stays byte-identical to the previous tool."""
    df = pd.read_csv(tsv_path, sep='\t')
    channels = list(pd.unique(df['channel']))               # epoch-major rows -> tool-6 channel order
    epoch_index = sorted(df['epoch_idx'].unique())
    stage_by_epoch = (df.drop_duplicates('epoch_idx').set_index('epoch_idx')['stage']
                      .astype(str).reindex(epoch_index))
    if methods is None:
        methods = [m for m in L.METHOD_ORDER if ('flag_' + m) in df.columns]
    piv = pd.DataFrame(False, index=epoch_index, columns=channels)

    def _or_flag(col):
        p = (df.pivot(index='epoch_idx', columns='channel', values=col)
             .reindex(index=epoch_index, columns=channels).fillna(0).astype(bool))
        return piv | p

    used_methods = []
    for m in methods:
        if m == 'event':
            continue                                        # event handled separately (epoch-level)
        if ('flag_' + m) in df.columns:
            piv = _or_flag('flag_' + m)
            used_methods.append(m)

    warn = None
    used_event_types = []
    if event_master and 'event' in methods:
        if event_flags_path is not None and Path(event_flags_path).exists():
            ef = pd.read_csv(event_flags_path, sep='\t').set_index('epoch_idx')
            evt_cols = [f'evt_{t}' for t in (event_types or []) if f'evt_{t}' in ef.columns]
            used_event_types = [c[len('evt_'):] for c in evt_cols]
            ev_mask = (ef[evt_cols].astype(bool).any(axis=1) if evt_cols
                       else pd.Series(False, index=ef.index))
            # broadcast the epoch-level event flag across all channels (matches flag_event's storage)
            ev_col = ev_mask.reindex(epoch_index).fillna(False).values[:, np.newaxis]
            ev_df = pd.DataFrame(np.repeat(ev_col, len(channels), axis=1),
                                 index=epoch_index, columns=channels)
            piv = piv | ev_df
            used_methods.append('event')
        elif 'flag_event' in df.columns:
            piv = _or_flag('flag_event')                    # old data: no per-type sidecar
            used_methods.append('event')
            warn = 'no _event_epoch_flags.tsv (used flag_event = all event types)'

    meta = {'used_methods': used_methods, 'used_event_types': used_event_types, 'warn': warn}
    return piv, stage_by_epoch, meta


def auto_reject_decision(piv, stage_by_epoch, stages_of_interest, ch_pct, ep_pct):
    """Channel-first then epoch decision (ch_pct / ep_pct are fractions in [0, 1]):
    (1) drop a channel flagged in > ch_pct of the in-scope epochs;
    (2) reject an in-scope epoch flagged in > ep_pct of the remaining GOOD channels."""
    channels = list(piv.columns)
    in_scope = stage_by_epoch.isin(stages_of_interest).values
    scope_epochs = list(piv.index[in_scope])
    M = piv.loc[scope_epochs]                                     # (n_scope, n_ch) bool
    n_scope = len(scope_epochs)
    badness = M.mean(axis=0) if n_scope else pd.Series(0.0, index=channels)
    rejected_channels = [c for c in channels if float(badness[c]) > ch_pct]
    good = [c for c in channels if c not in rejected_channels]
    if good and n_scope:
        frac = M[good].mean(axis=1)
    else:
        frac = pd.Series(1.0, index=scope_epochs)                # no good channel -> reject all in-scope
    reject_epoch = frac > ep_pct
    kept_epochs = [int(e) for e, r in reject_epoch.items() if not bool(r)]
    return {
        'channels': channels, 'badness': badness, 'rejected_channels': rejected_channels,
        'good_channels': good, 'scope_epochs': [int(e) for e in scope_epochs], 'n_scope': n_scope,
        'reject_epoch': reject_epoch, 'kept_epochs': kept_epochs,
        'n_rejected_epochs': int(reject_epoch.sum()), 'stage_by_epoch': stage_by_epoch,
    }


def decision_summary_row(fid, dec, soi, ch_pct, ep_pct, methods_used='', event_types_used='',
                         excluded=False, exclude_reason=''):
    """One-row DataFrame = the durable per-participant record AND the global-summary row source.
    methods_used / event_types_used trace which flagging methods (and event types) drove the decision;
    excluded / exclude_reason mark a participant dropped by a per-event count threshold."""
    badness = dec['badness']
    badness_str = ';'.join(f'{c}:{float(badness[c]):.2f}' for c in dec['channels'])
    n_scope, n_rej = dec['n_scope'], dec['n_rejected_epochs']
    return pd.DataFrame([{
        'file_id': fid,
        'n_channels': len(dec['channels']),
        'n_channels_rejected': len(dec['rejected_channels']),
        'rejected_channels': ';'.join(dec['rejected_channels']),
        'good_channels': ';'.join(dec['good_channels']),
        'channel_badness_pct': badness_str,
        'channel_reject_pct': round(ch_pct * 100, 3),
        'epoch_reject_pct': round(ep_pct * 100, 3),
        'stages_of_interest': '+'.join(soi),
        'methods_used': methods_used,
        'event_types_used': event_types_used,
        'excluded': bool(excluded),
        'exclude_reason': exclude_reason,
        'n_epochs': n_scope,
        'n_epochs_rejected': n_rej,
        'n_epochs_kept': len(dec['kept_epochs']),
        'pct_epochs_rejected': round(100 * n_rej / n_scope, 2) if n_scope else 0.0,
    }])


def wrap_scroll(html_table):
    """Wrap a wide HTML table so it scrolls horizontally inside its container instead of overflowing
    the page to the right (applied to the notebook display and the global HTML report)."""
    return ('<div style="overflow-x:auto; max-width:100%">'
            + html_table.replace('<table', '<table style="border-collapse:collapse"', 1)
            + '</div>')


def per_stage_table_html(dec, soi):
    """HTML: rejected / kept epoch counts per stage within the selected stages."""
    sbe, rej = dec['stage_by_epoch'], dec['reject_epoch']
    rows = []
    for st in soi:
        st_epochs = [e for e in dec['scope_epochs'] if sbe[e] == st]
        n = len(st_epochs); r = sum(int(rej[e]) for e in st_epochs)
        pct = f'{100 * r / n:.1f}%' if n else '-'
        rows.append(f'<tr><td>{st}</td><td>{n}</td><td>{r}</td><td>{pct}</td></tr>')
    return ('<table border="1" cellpadding="4" style="border-collapse:collapse">'
            '<tr><th>Stage</th><th>Scope epochs</th><th>Rejected</th><th>% rejected</th></tr>'
            + ''.join(rows) + '</table>')


def decision_overview_html(fid, dec, soi, ch_pct, ep_pct, methods_used='', event_types_used='',
                           exclude_reason=''):
    """HTML summary block for the per-participant report. Traces the flagging methods (and event types)
    that drove the decision, and shows a red banner when the participant was excluded by an event
    count threshold."""
    badness = dec['badness']
    ch_lines = ''.join(
        f'<li>{c}: {100 * float(badness[c]):.1f}% flagged'
        + (' <b style="color:#c0392b">-> dropped</b>' if c in dec['rejected_channels'] else '')
        + '</li>' for c in dec['channels'])
    pct = 100 * dec['n_rejected_epochs'] / dec['n_scope'] if dec['n_scope'] else 0.0
    banner = (f'<p style="padding:6px 10px;background:#fdecea;border:1px solid #c0392b;color:#c0392b;">'
              f'<b>EXCLUDED</b> &mdash; {exclude_reason}. No clean-epo written.</p>') if exclude_reason else ''
    methods_line = (f'<p>Methods used: <b>{methods_used or "(none)"}</b>'
                    + (f' &mdash; event types: <b>{event_types_used}</b>' if event_types_used else '')
                    + '</p>')
    return (
        f'<h3>{fid} - automatic rejection</h3>'
        + banner + methods_line +
        f'<p>Stages of interest: <b>{"+".join(soi)}</b> &mdash; channel threshold &gt; {100 * ch_pct:.0f}% '
        f'&mdash; epoch threshold &gt; {100 * ep_pct:.0f}% of good channels.</p>'
        f'<p>Channels ({len(dec["good_channels"])}/{len(dec["channels"])} kept):</p><ul>{ch_lines}</ul>'
        f'<p>In-scope epochs: <b>{dec["n_scope"]}</b> &mdash; rejected: <b>{dec["n_rejected_epochs"]}</b> '
        f'&mdash; kept: <b>{len(dec["kept_epochs"])}</b> ({pct:.1f}% rejected)</p>'
        + per_stage_table_html(dec, soi))


def plot_decision_heatmap(piv, dec, fid, custom_stages):
    """Channels x epochs flagged-pair heatmap with a hypnogram strip and a rejected-epoch strip.
    Rejected-channel rows get a red bold label; out-of-scope epochs are greyed in the reject strip."""
    channels = dec['channels']
    epochs = list(piv.index)
    ep_pos = {e: i for i, e in enumerate(epochs)}
    n_ep, n_ch = len(epochs), len(channels)
    M = piv[channels].values.T.astype(float)                     # (n_ch, n_ep) 0/1

    stage_y, _sc, ytick_pos, ytick_labels = L.custom_stage_style(list(custom_stages))
    sbe = dec['stage_by_epoch']
    hyp = np.array([stage_y.get(str(sbe[e]), np.nan) for e in epochs], dtype=float)

    scope_set = set(dec['scope_epochs']); rej = dec['reject_epoch']
    strip = np.full(n_ep, 2.0)                                    # 2 = out of scope (grey)
    for e in epochs:
        if e in scope_set:
            strip[ep_pos[e]] = 1.0 if bool(rej[e]) else 0.0      # 1 = rejected (red), 0 = kept (green)

    # The channels x epochs matrix height scales with the channel count (like tool 6's heatmap); the
    # hypnogram and epoch-decision strips keep a FIXED height so they stay readable on dense montages.
    fig_w   = max(8, min(n_ep / 6, 20))
    h_hyp   = 1.0                                                 # inches, fixed
    h_strip = 0.4                                                 # inches, fixed
    h_mat   = min(max(0.22 * n_ch, 0.8), 12.0)                   # inches, ~0.22 in / channel row
    fig, (ax_h, ax_s, ax_m) = plt.subplots(
        3, 1, figsize=(fig_w, h_hyp + h_strip + h_mat + 0.7), sharex=True, layout='constrained',
        gridspec_kw={'height_ratios': [h_hyp, h_strip, h_mat]})

    ax_h.step(range(n_ep), hyp, where='mid', color='#333333', lw=0.8)
    ax_h.set_yticks(ytick_pos); ax_h.set_yticklabels(ytick_labels, fontsize=7)
    ax_h.set_ylabel('stage', fontsize=8)
    ax_h.set_title(f'{fid} - flagged (epoch x channel); red = dropped channel / rejected epoch', fontsize=10)

    ax_s.imshow(strip[np.newaxis, :], aspect='auto', vmin=0, vmax=2,
                cmap=ListedColormap(['#2e7d32', '#c0392b', '#d9d9d9']))
    ax_s.set_yticks([]); ax_s.set_ylabel('epoch', fontsize=8, rotation=0, ha='right', va='center')

    # 0 = not flagged, 1 = flagged (good channel, blue), 2 = flagged in a DROPPED channel (grey).
    # Only the flagged cells of a dropped channel turn grey, so which epochs were flagged stays visible.
    disp = M.copy()
    for i, c in enumerate(channels):
        if c in dec['rejected_channels']:
            disp[i, M[i, :] > 0] = 2.0
    ax_m.imshow(disp, aspect='auto', vmin=0, vmax=2, interpolation='nearest',
                cmap=ListedColormap(['#f7f7f7', '#34495e', '#d9d9d9']))
    ax_m.set_yticks(range(n_ch)); ax_m.set_yticklabels(channels, fontsize=8)
    for tick, c in zip(ax_m.get_yticklabels(), channels):
        if c in dec['rejected_channels']:
            tick.set_color('#c0392b'); tick.set_fontweight('bold')
    ax_m.set_xlabel('epoch index', fontsize=8)
    return fig


## Section 1 - Select the folders

Optionally set the **data folder** first (holds `derivatives/` + `reports_preprocessing/`, as in tool 6) —
it just pre-points the next two choosers. Then pick the **raw-epochs folder** (holds the tool-6
`{file_id}_all-epo.fif`, e.g. `derivatives/raw_epo`) and the **reports folder** (holds
`{file_id}_epoch_channel_rejection.tsv`, e.g. `reports_preprocessing`), and click **Scan**. Selecting the
two folders explicitly keeps renamed/versioned tool-6 runs apart. The clean `*_clean-epo.fif` is written to
`clean_epo_auto/` (beside the raw folder); the reports go to `reports_rejection_auto/` (beside the reports
folder).

In [ ]:
fc_data = FileChooser(os.getcwd())
fc_data.show_only_dirs = True
fc_data.title = ('<b>Data folder</b> (holds <code>derivatives/</code> + '
                 '<code>reports_preprocessing/</code>) &mdash; optional, pre-points the two folders below:')

fc_raw = FileChooser(os.getcwd())
fc_raw.show_only_dirs = True
fc_raw.title = '<b>Raw-epochs folder</b> (holds the tool-6 <code>*_all-epo.fif</code>):'

fc_reports = FileChooser(os.getcwd())
fc_reports.show_only_dirs = True
fc_reports.title = '<b>Reports folder</b> (holds <code>*_epoch_channel_rejection.tsv</code>):'

btn_scan = widgets.Button(description='Scan', button_style='primary', icon='search')
lbl_scan = widgets.HTML('<i>Optionally set the data folder, pick the raw-epochs and reports folders, then Scan.</i>')


def _on_data(chooser):
    # Convenience: point fc_raw into <data>/derivatives (where raw_epo* live) and fc_reports at <data>
    # (the parent of reports_preprocessing*). The user still picks the specific (possibly versioned) folder.
    try:
        data = fc_data.selected_path
        if not data:
            return
        data = Path(data)
        raw_start = data / 'derivatives' if (data / 'derivatives').is_dir() else data
        fc_raw.reset(path=str(raw_start))
        fc_raw.title = '<b>Raw-epochs folder</b> (holds the tool-6 <code>*_all-epo.fif</code>):'
        fc_reports.reset(path=str(data))
        fc_reports.title = '<b>Reports folder</b> (holds <code>*_epoch_channel_rejection.tsv</code>):'
        lbl_scan.value = ('<i>Data folder set &mdash; pick the raw-epochs folder (inside '
                          '<code>derivatives/</code>) and the reports folder, then Scan.</i>')
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Data folder error: {e}</span>'


def _scan(b):
    try:
        raw, reports = fc_raw.selected_path, fc_reports.selected_path
        if not raw or not reports:
            lbl_scan.value = '<span style="color:#c62828">Select both folders first.</span>'; return
        raw_root, reports_root = Path(raw), Path(reports)
        parts = L.find_participants(raw_root)
        # Locate each participant's flags TSV + the optional event sidecars in the chosen reports
        # folder (once; reused by the run). Also collect the methods present and the event types.
        tsv_by_fid = {}
        flags_by_fid = {}      # {fid: _event_epoch_flags.tsv path or None}
        counts_by_fid = {}     # {fid: _event_counts.tsv path or None}
        stages = set(); methods_present = set(); event_types = set()
        type_counts = {}       # {event_type: [n_events per file that has it]} -> mean/median hints
        for p in parts:
            fid = p['file_id']
            tsv = next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)
            tsv_by_fid[fid] = tsv
            if tsv is not None:
                try:
                    cols = pd.read_csv(tsv, sep='\t', nrows=0).columns
                    methods_present.update(m for m in L.METHOD_ORDER if ('flag_' + m) in cols)
                    stages.update(pd.read_csv(tsv, sep='\t', usecols=['stage'])['stage'].astype(str).unique())
                except Exception:
                    pass
            ef = next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
            flags_by_fid[fid] = ef
            if ef is not None:
                try:
                    event_types.update(c[len('evt_'):] for c in pd.read_csv(ef, sep='\t', nrows=0).columns
                                       if c.startswith('evt_'))
                except Exception:
                    pass
            ec = next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
            counts_by_fid[fid] = ec
            if ec is not None:
                try:
                    ecdf = pd.read_csv(ec, sep='\t')
                    event_types.update(ecdf['event_type'].astype(str).unique())
                    for _, r in ecdf.iterrows():
                        type_counts.setdefault(str(r['event_type']), []).append(int(r['n_events']))
                except Exception:
                    pass
        ordered = [s for s in AASM_STAGES if s in stages] + sorted(s for s in stages if s not in AASM_STAGES)
        S['raw_root'] = raw_root; S['reports_root'] = reports_root
        S['parts'] = parts; S['tsv_by_fid'] = tsv_by_fid
        S['flags_by_fid'] = flags_by_fid; S['counts_by_fid'] = counts_by_fid
        # ---- Stage checkboxes (default ticked: N2/N3/R, else all) ----
        prefer = [s for s in ordered if s in ('N2', 'N3', 'R')]
        S['stage_checks'] = {}
        boxes = []
        for st in ordered:
            cb = widgets.Checkbox(value=(st in prefer) if prefer else True, description=st,
                                  indent=False, layout=widgets.Layout(width='75px'))
            S['stage_checks'][st] = cb
            boxes.append(cb)
        stage_box.children = boxes
        # ---- Flagging-method checkboxes (all ON by default = today's reject_any) ----
        S['method_checks'] = {}
        mboxes = []
        for m in [m for m in L.METHOD_ORDER if m in methods_present]:
            cb = widgets.Checkbox(value=True, description=L.METHOD_LABEL.get(m, m),
                                  indent=False, layout=widgets.Layout(width='120px'))
            S['method_checks'][m] = cb
            mboxes.append(cb)
        method_box.children = mboxes
        # ---- Event-type rows (checkbox + per-type exclusion threshold), nested under the 'event' method ----
        S['event_type_rows'] = {}
        erows = []
        for t in sorted(event_types):
            cbt = widgets.Checkbox(value=True, description=t, indent=False,
                                   layout=widgets.Layout(width='190px'))
            thr = widgets.IntText(value=0, layout=widgets.Layout(width='80px'))
            # Per-file event-count hint across the files that HAVE this type (guides the threshold).
            vals = type_counts.get(t, [])
            hint = (f'&nbsp;<small style="color:#666;">mean {np.mean(vals):.1f} (median '
                    f'{np.median(vals):g}) events/file, n={len(vals)}</small>') if vals else ''
            erows.append(widgets.HBox([cbt, widgets.HTML('exclude participant if &gt;'), thr,
                                       widgets.HTML('events (0 = off)'), widgets.HTML(hint)]))
            S['event_type_rows'][t] = (cbt, thr)
        event_types_box.children = erows
        # Reveal the event-type rows only when the 'event' method is ticked (initial display from its value).
        ev_master = S['method_checks'].get('event')
        if ev_master is not None:
            def _toggle_events(change=None):
                event_section.layout.display = '' if ev_master.value else 'none'
            ev_master.observe(_toggle_events, names='value')
            _toggle_events()
        else:
            event_section.layout.display = 'none'   # no event flagging in any scanned file
        n_tsv = sum(1 for t in tsv_by_fid.values() if t is not None)
        colour = '#2e7d32' if (parts and n_tsv == len(parts)) else ('#c62828' if not parts else '#e69500')
        msg = f'{len(parts)} participant(s) in the raw folder; {n_tsv} with a matching TSV in the reports folder.'
        if parts and n_tsv < len(parts):
            msg += ' &#9888; some participants have no TSV here (version mismatch?).'
        lbl_scan.value = (f'<span style="color:{colour}">{msg}</span>'
                          f'<br><small>clean .fif &rarr; {raw_root.parent / "clean_epo_auto"}'
                          f'<br>reports &rarr; {reports_root.parent / "reports_rejection_auto"}</small>')
    except Exception as e:
        lbl_scan.value = f'<span style="color:#c62828">Scan error: {e}</span>'


fc_data.register_callback(_on_data)
btn_scan.on_click(_scan)
display(widgets.VBox([fc_data, fc_raw, fc_reports, btn_scan, lbl_scan]))


## Section 2 - Methods, thresholds & stages of interest

A **channel** is dropped when flagged in more than *Channel reject %* of its in-scope epochs; then an
**epoch** is rejected when flagged in more than *Epoch reject %* of the remaining **good** channels. Both
default to 20% and are editable. The decision is computed over the **stages of interest** only, and the
output `*_clean-epo.fif` keeps only those stages.

**Flagging methods** (populated at Scan): tick which tool-6 methods drive the rejection — all on
reproduces the previous behaviour, but after inspecting the tool-6 flagging reports you can drop a method
that looks unhelpful. Ticking **event** reveals the **event types**: sub-select which types may reject an
epoch, and set a per-type threshold to **exclude the whole participant** when the file contains more raw
events of that type than the threshold (0 = no exclusion). Event sub-selection needs the tool-6
`*_event_epoch_flags.tsv` / `*_event_counts.tsv` sidecars; files preprocessed before this feature fall
back to the whole `flag_event` (all event types, no sub-selection).

In [ ]:
ft_ch_pct = widgets.FloatText(value=DEFAULT_CHANNEL_REJECT_PCT, description='Channel reject > (%)',
                              style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))
ft_ep_pct = widgets.FloatText(value=DEFAULT_EPOCH_REJECT_PCT, description='Epoch reject > (%)',
                              style={'description_width': '150px'}, layout=widgets.Layout(width='250px'))
# All three boxes below are populated when you Scan (Section 1): one checkbox per detected stage,
# one per flagging method present, and one row (checkbox + exclusion threshold) per event type.
stage_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap',
                                                  width='100%'))
method_box = widgets.Box([], layout=widgets.Layout(display='flex', flex_flow='row wrap', width='100%'))
# Nested under the 'event' method checkbox: revealed only when 'event' is ticked (wired at Scan time).
event_types_box = widgets.VBox([], layout=widgets.Layout(margin='2px 0 0 25px'))
event_section = widgets.VBox([
    widgets.HTML('<small><b>Event types</b> &mdash; tick the types allowed to reject an epoch; set a '
                 'threshold to <b>exclude the whole participant</b> when the file has more events of that '
                 'type than the threshold (0 = no exclusion):</small>'),
    event_types_box,
])
cb_skip = widgets.Checkbox(value=True, description='Skip already processed', indent=False)

display(widgets.VBox([
    ft_ch_pct, ft_ep_pct,
    widgets.HTML('<b>Flagging methods</b> &mdash; tick the methods used to decide rejection '
                 '(all on = reproduce the previous behaviour; untick a method after reviewing the '
                 'tool-6 reports to drop it):'),
    method_box, event_section,
    widgets.HTML('<b>Stages of interest</b> &mdash; tick the stages the cleaned epochs are meant for '
                 '(channel-badness and epoch rejection are computed on these only):'),
    stage_box, cb_skip,
]))


## Section 3 - Run

Loops over every participant found by Scan (Section 1). The clean `{file_id}_clean-epo.fif` goes to
`clean_epo_auto/<subtree>/` (beside the raw folder); `{file_id}_autoreject_decision.tsv` +
`{file_id}_autoreject_report.html` + the rebuilt `global_autoreject_summary.tsv` go to
`reports_rejection_auto/` (beside the reports folder). A **summary table** is shown at the end. A
participant is
**skipped** if it already has both a clean-epo and a decision TSV (uncheck *Skip* to reprocess).
Participants where **every channel** (or every in-scope epoch) would be rejected still get a decision row
and a report (shown as **100% rejected** in the summary + bar chart) — only the clean-epo is not written
(nothing survives to save). A participant **excluded** by an event-count threshold likewise gets a
decision row + report (flagged *EXCLUDED* with the reason) but no clean-epo.

In [ ]:
btn_run = widgets.Button(description='Run automatic rejection', button_style='success', icon='play')
prog = widgets.IntProgress(description='Participants:', min=0, max=1, value=0,
                           layout=widgets.Layout(width='430px'))
lbl_prog = widgets.HTML(layout=widgets.Layout(margin='0 0 0 8px'))   # current participant (i/N)
out_run = widgets.Output()


def _rebuild_global_summary(auto_root):
    files = sorted(Path(auto_root).rglob('*_autoreject_decision.tsv'))
    rows = []
    for f in files:
        try:
            rows.append(pd.read_csv(f, sep='\t'))
        except Exception:
            pass
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def run_autoreject(b):
    with out_run:
        clear_output(wait=True)
        try:
            if 'raw_root' not in S:
                print('Select the raw + reports folders and click Scan first (Section 1).'); return
            raw_root = Path(S['raw_root']); reports_root = Path(S['reports_root'])
            soi = [st for st, cb in S.get('stage_checks', {}).items() if cb.value]
            if not soi:
                print('Tick at least one stage of interest (Section 2).'); return
            ch_pct = float(ft_ch_pct.value) / 100.0
            ep_pct = float(ft_ep_pct.value) / 100.0
            skip = cb_skip.value
            # Section-2 flagging-method + event sub-selection.
            methods_sel = [m for m, cb in S.get('method_checks', {}).items() if cb.value]
            event_master_on = ('event' in methods_sel)
            event_types_sel = [t for t, (cb, thr) in S.get('event_type_rows', {}).items() if cb.value]
            # Per-type exclusion thresholds: only ticked types with a positive threshold are evaluated.
            event_thresholds = {t: int(thr.value) for t, (cb, thr) in S.get('event_type_rows', {}).items()
                                if cb.value and int(thr.value) > 0}
            if not methods_sel:
                print('Tick at least one flagging method (Section 2).'); return
            flags_by_fid = S.get('flags_by_fid', {}); counts_by_fid = S.get('counts_by_fid', {})
            parts = S.get('parts', [])
            tsv_by_fid = S.get('tsv_by_fid', {})
            # DATA (the clean .fif) beside the raw-epochs folder; REPORTS (decision tsv + html + globals)
            # beside the reports folder — following the toolkit's derivatives/ vs reports_* split.
            data_root   = raw_root.parent / 'clean_epo_auto'
            reports_out = reports_root.parent / 'reports_rejection_auto'
            prog.max = max(1, len(parts)); prog.value = 0
            failed = []; saved = 0; all_rejected = 0; skipped = 0; excluded = 0
            for i, p in enumerate(parts):
                fid = p['file_id']
                lbl_prog.value = f'<b>{fid}</b> ({i + 1}/{len(parts)})'
                try:
                    try:
                        subtree = Path(p['folder']).relative_to(raw_root)   # participant subtree in the raw folder
                    except Exception:
                        subtree = Path('.')
                    data_dir    = data_root / subtree                       # holds the clean .fif
                    reports_dir = reports_out / subtree                     # holds the decision tsv + html
                    clean_fif    = data_dir / f'{fid}_clean-epo.fif'
                    decision_tsv = reports_dir / f'{fid}_autoreject_decision.tsv'
                    if skip and clean_fif.exists() and decision_tsv.exists():
                        skipped += 1; prog.value += 1; continue
                    tsv = tsv_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_epoch_channel_rejection.tsv'), None)
                    if tsv is None:
                        failed.append((fid, 'no _epoch_channel_rejection.tsv in the reports folder'))
                        prog.value += 1; continue
                    ef_path = flags_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_epoch_flags.tsv'), None)
                    ec_path = counts_by_fid.get(fid) or next(reports_root.rglob(f'{fid}_event_counts.tsv'), None)
                    piv, stage_by_epoch, pm_meta = build_pair_matrix(
                        tsv, methods=methods_sel, event_master=event_master_on,
                        event_types=event_types_sel, event_flags_path=ef_path)
                    if pm_meta.get('warn'):
                        print(f'  ⚠ {fid}: {pm_meta["warn"]}')
                    methods_used_str = '+'.join(pm_meta['used_methods']) or '(none)'
                    event_types_used_str = '+'.join(pm_meta['used_event_types'])
                    stages_present = set(stage_by_epoch.dropna().astype(str).unique())
                    soi_p = [s for s in soi if s in stages_present]
                    if not soi_p:
                        failed.append((fid, 'none of the selected stages present')); prog.value += 1; continue
                    dec = auto_reject_decision(piv, stage_by_epoch, soi_p, ch_pct, ep_pct)
                    # Per-event participant exclusion: drop the WHOLE participant when the file has more
                    # events of a ticked type than its threshold (raw event count from _event_counts.tsv).
                    excl_reason = ''
                    if event_master_on and event_thresholds:
                        counts = read_event_counts(ec_path) if ec_path is not None else {}
                        for t, thr in event_thresholds.items():
                            n = counts.get(t)
                            if n is not None and n > thr:
                                excl_reason = f'{n} {t} events > {thr}'
                                break
                    # An all-rejected participant (every channel dropped, or every in-scope epoch rejected)
                    # is a valid extreme outcome: still write the decision row + report (so it shows up as
                    # 100% rejected in the summary), just no clean-epo (nothing survives to save). An
                    # excluded participant likewise gets a row + report but no clean-epo.
                    has_output = (not excl_reason) and bool(dec['good_channels']) and len(dec['kept_epochs']) > 0
                    reports_dir.mkdir(parents=True, exist_ok=True)
                    if has_output:
                        data_dir.mkdir(parents=True, exist_ok=True)
                        epochs = mne.read_epochs(str(p['fif']), preload=True, verbose=False)
                        meta_idx = epochs.metadata['epoch_idx'].astype(int).tolist()
                        keep_set = set(dec['kept_epochs'])
                        keep_pos = [i for i, e in enumerate(meta_idx) if e in keep_set]
                        clean = epochs[keep_pos]
                        drop = [c for c in dec['rejected_channels'] if c in clean.ch_names]
                        if drop:
                            clean.drop_channels(drop)
                        meta = clean.metadata.copy()
                        meta['auto_reject_channel_pct'] = 100 * ch_pct
                        meta['auto_reject_epoch_pct'] = 100 * ep_pct
                        meta['auto_reject_stages'] = '+'.join(soi_p)
                        meta['auto_reject_methods'] = methods_used_str
                        meta['auto_reject_event_types'] = event_types_used_str
                        clean.metadata = meta
                        clean.save(str(clean_fif), overwrite=True, verbose=False)
                    # --- decision TSV (always: feeds the summary, incl. all-rejected / excluded participants) ---
                    decision_summary_row(fid, dec, soi_p, ch_pct, ep_pct,
                                         methods_used=methods_used_str, event_types_used=event_types_used_str,
                                         excluded=bool(excl_reason), exclude_reason=excl_reason).to_csv(
                        decision_tsv, sep='\t', index=False)
                    # --- per-participant report (always, non-fatal) ---
                    try:
                        custom = [s for s in stages_present if s not in AASM_STAGES]
                        fig = plot_decision_heatmap(piv, dec, fid, custom)
                        html = decision_overview_html(fid, dec, soi_p, ch_pct, ep_pct,
                                                      methods_used=methods_used_str,
                                                      event_types_used=event_types_used_str,
                                                      exclude_reason=excl_reason)
                        L.save_report_html(reports_dir / f'{fid}_autoreject_report.html',
                                           f'{fid} - automatic rejection', [('Decision', fig)], html)
                    except Exception as rep_e:
                        print(f'  ⚠ {fid}: report failed ({rep_e}).')
                    if excl_reason:
                        excluded += 1
                    elif has_output:
                        saved += 1
                    else:
                        all_rejected += 1
                except Exception as e:
                    failed.append((fid, str(e)))
                prog.value += 1
            lbl_prog.value = f'done ({len(parts)}/{len(parts)})'
            # --- global summary (rebuilt from disk) + failures --- (all in the reports folder)
            summary = _rebuild_global_summary(reports_out)
            if not summary.empty:
                summary = summary.sort_values('file_id')
                reports_out.mkdir(parents=True, exist_ok=True)
                summary.to_csv(reports_out / 'global_autoreject_summary.tsv', sep='\t', index=False)
            if failed:
                reports_out.mkdir(parents=True, exist_ok=True)
                pd.DataFrame(failed, columns=['file_id', 'reason']).to_csv(
                    reports_out / 'autoreject_failed.tsv', sep='\t', index=False)
            print(f'Done. saved={saved}  all-rejected (no clean-epo)={all_rejected}  '
                  f'excluded (event threshold)={excluded}  skipped={skipped}  failed={len(failed)}')
            print(f'  clean .fif -> {data_root}')
            print(f'  reports    -> {reports_out}')
            if failed:
                print('Failed (could not process):')
                for fid, why in failed:
                    print(f'  ⚠ {fid}: {why}')
            # --- end-of-run summary table + global report ---
            if not summary.empty:
                # Compact inline table (the full table stays in the global + per-participant reports).
                disp = summary[['file_id', 'n_channels', 'n_channels_rejected',
                                'n_epochs', 'n_epochs_rejected']].copy()
                disp['n_channels_rejected'] = summary.apply(
                    lambda r: (f"{int(r['n_channels_rejected'])} "
                               f"({100 * r['n_channels_rejected'] / r['n_channels']:.1f}%)")
                    if r['n_channels'] else str(int(r['n_channels_rejected'])), axis=1)
                disp['n_epochs_rejected'] = summary.apply(
                    lambda r: f"{int(r['n_epochs_rejected'])} ({r['pct_epochs_rejected']:.1f}%)", axis=1)
                # Show an 'excluded' column only when a participant was dropped by an event threshold.
                if 'excluded' in summary.columns and summary['excluded'].fillna(False).astype(bool).any():
                    disp['excluded'] = summary['exclude_reason'].fillna('').values
                display(HTML('<h3>Summary - automatic rejection</h3>' + disp.to_html(index=False)))
                try:
                    fig, ax = plt.subplots(figsize=(max(6, min(14, 2 + 0.5 * len(summary))), 3.6),
                                           layout='constrained')
                    ax.bar(summary['file_id'].astype(str), summary['pct_epochs_rejected'], color='#34495e')
                    ax.set_ylabel('% in-scope epochs rejected'); ax.set_xlabel('participant')
                    ax.set_ylim(0, 100)
                    ax.set_title('% in-scope epochs rejected per participant')
                    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
                    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
                    display(widgets.Image(value=buf.getvalue(), format='png'))
                    L.save_report_html(reports_out / 'autoreject_database_report.html',
                                       'Automatic rejection - database summary',
                                       [('% in-scope epochs rejected per participant', fig)],
                                       wrap_scroll(summary.to_html(index=False)))
                except Exception as ge:
                    print(f'⚠ global chart/report failed: {ge}')
        except Exception as e:
            print(f'Run error: {e}')


btn_run.on_click(run_autoreject)
display(widgets.VBox([btn_run, widgets.HBox([prog, lbl_prog]), out_run]))
